# Ecommerce Runner

This notebook is the Jupyter equivalent of the `runner.py` script.

### Setup: Imports & Environment
Imports the libraries used throughout the notebook (Graphiti core classes, the Anthropic LLM client, async/datetime helpers, and `rich` for pretty output) and loads environment variables from `.env` via `load_dotenv()`.

In [1]:
import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from rich.pretty import pprint

from graphiti_core import Graphiti
from graphiti_core.edges import EntityEdge
from graphiti_core.llm_client.anthropic_client import AnthropicClient
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.nodes import EpisodeType
from graphiti_core.utils.bulk_utils import RawEpisode
from graphiti_core.utils.maintenance.graph_data_operations import clear_data

load_dotenv()

True

### Neo4j Connection Settings
Reads the Neo4j connection URI, username, and password from environment variables (falling back to local defaults) — these get passed to `Graphiti` later to open the driver.

In [2]:
neo4j_uri = os.environ.get('NEO4J_URI', 'bolt://localhost:7687')
neo4j_user = os.environ.get('NEO4J_USER', 'neo4j')
neo4j_password = os.environ.get('NEO4J_PASSWORD', 'password')

### Logging
Configures the root logger. Kept at `WARNING` (not `INFO`) so verbose `httpx`/`neo4j`/`anthropic` request logs don't interleave with `rich`'s pretty-printed search output below.

In [3]:
def setup_logging():
    logger = logging.getLogger()
    logger.setLevel(logging.WARNING)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.WARNING)
    formatter = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger


logger = setup_logging()

### Sample Conversation Data
Two scripted shoe-shopping conversations between a customer ("John") and a sales bot, used later to demonstrate episode ingestion from natural-language messages.

In [4]:
shoe_conversation_1 = [
    "SalesBot (2024-07-30T00:00:00Z): Hi, I'm ManyBirds Assistant! How can I help you today?",
    "John (2024-07-30T00:01:00Z): Hi, I'm looking for a new pair of shoes.",
    'SalesBot (2024-07-30T00:02:00Z): Of course! What kind of material are you looking for?',
    "John (2024-07-30T00:03:00Z): I'm allergic to wool. Also, I'm a size 10 if that helps?",
    "SalesBot (2024-07-30T00:04:00Z): We have just what you are looking for, how do you like our Men's Couriers. They have a retro silhouette look and from cotton. How about them in Basin Blue?",
    "John (2024-07-30T00:05:00Z): Blue is great! Love the look. I'll take them.",
]

shoe_conversation_2 = [
    'SalesBot (2024-08-20T00:00:00Z): Hi John, how can I assist you today?',
    "John (2024-08-20T00:01:00Z): Hi, I need to return the Men's Couriers I bought recently. They're too tight for my wide feet. Hahaha.",
    "SalesBot (2024-08-20T00:02:00Z): I'm sorry to hear that. We can process the return for you.",
]

### Helper: Ingest a Conversation
`add_messages` feeds a list of conversation lines into the graph one at a time via `client.add_episode`, each becoming its own timestamped episode.

In [5]:
async def add_messages(client: Graphiti, messages: list[str], prefix: str = 'Message'):
    for i, message in enumerate(messages):
        await client.add_episode(
            name=f'{prefix}-{i}',
            episode_body=message,
            source=EpisodeType.message,
            reference_time=datetime.now(timezone.utc),
            source_description='Shoe conversation',
        )

### Helper: Ingest Product Catalog
`ingest_products_data` loads `data/manybirds_products.json` and bulk-ingests each product as a JSON episode via `client.add_episode_bulk`, giving the graph its initial catalog of shoes.

In [6]:
async def ingest_products_data(client: Graphiti):
    script_dir = Path.cwd().parent
    json_file_path = script_dir / 'data' / 'manybirds_products.json'

    with open(json_file_path) as file:
        products = json.load(file)['products']

    episodes: list[RawEpisode] = [
        RawEpisode(
            name=product.get('title', f'Product {i}'),
            content=str({k: v for k, v in product.items() if k != 'images'}),
            source_description='ManyBirds products',
            source=EpisodeType.json,
            reference_time=datetime.now(timezone.utc),
        )
        for i, product in enumerate(products)
    ]

    await client.add_episode_bulk(episodes)

### Helper: Pretty-Print Search Results
`pretty_print` renders an `EntityEdge` (or list of them) with `rich`, stripping the bulky `fact_embedding` vector so the output stays readable.

In [4]:
def pretty_print(entity: EntityEdge | list[EntityEdge]):
    if isinstance(entity, EntityEdge):
        data = {k: v for k, v in entity.model_dump().items() if k != 'fact_embedding'}
    elif isinstance(entity, list):
        data = [{k: v for k, v in e.model_dump().items() if k != 'fact_embedding'} for e in entity]
    else:
        pprint(entity)
        return
    pprint(data)

### Create the Graphiti Client
Instantiates the Anthropic LLM client (model pinned to a Sonnet snapshot so the prompt-caching setup above actually takes effect) and wraps it in a `Graphiti` client connected to the local Neo4j instance.

In [5]:
llm_client = AnthropicClient(
    config=LLMConfig(model='claude-sonnet-4-5-20250929'), cache=False
)

client = Graphiti(
    neo4j_uri,
    neo4j_user,
    neo4j_password,
    llm_client=llm_client,
)

### Reset & Seed the Graph
Wipes any existing graph data, (re)creates the required Neo4j indices/constraints, then bulk-ingests the full product catalog via `ingest_products_data`. This cell issues one LLM extraction call per product, so expect it to take a while.

In [9]:
await clear_data(client.driver)
await client.build_indices_and_constraints()
await ingest_products_data(client)

### Add an Inventory Update Episode
Inserts a single free-text episode noting that Tinybirds Wool Runners are out of stock — lets you test how a new fact updates/temporally bounds an existing part of the graph.

In [10]:
await client.add_episode(
    name='Inventory management 0',
    episode_body=('All Tinybirds Wool Runners styles are out of stock until December 25th 2024'),
    source=EpisodeType.text,
    reference_time=datetime.now(timezone.utc),
    source_description='Inventory Management Bot',
)

AddEpisodeResults(episode=EpisodicNode(uuid='f8161f44-ab4c-4f7b-8c30-665969685e4c', name='Inventory management 0', group_id='', labels=[], created_at=datetime.datetime(2026, 7, 27, 17, 14, 36, 13824, tzinfo=datetime.timezone.utc), source=<EpisodeType.text: 'text'>, source_description='Inventory Management Bot', content='All Tinybirds Wool Runners styles are out of stock until December 25th 2024', valid_at=datetime.datetime(2026, 7, 27, 17, 14, 36, 13810, tzinfo=datetime.timezone.utc), entity_edges=[], episode_metadata=None), episodic_edges=[EpisodicEdge(uuid='5bf096c5-261b-43da-97a6-2d4d4870ec6e', group_id='', source_node_uuid='f8161f44-ab4c-4f7b-8c30-665969685e4c', target_node_uuid='c2bd3223-7587-4bc5-9cae-794f61a80720', created_at=datetime.datetime(2026, 7, 27, 17, 14, 36, 13824, tzinfo=datetime.timezone.utc))], nodes=[EntityNode(uuid='c2bd3223-7587-4bc5-9cae-794f61a80720', name='TinyBirds Wool Runners - Little Kids - Natural Black (Blizzard Sole)', group_id='', labels=['Entity'], cr

### Search: Out-of-Stock Products
Runs a basic hybrid search (`client.search`) for out-of-stock products and prints the top matching fact — confirms the inventory episode above was picked up.

In [11]:
r = await client.search('Which products are out of stock?')

pretty_print(r[0])

{
│   'uuid': '8f042d94-49c2-4028-b268-1e811442b116',
│   'group_id': '',
│   'source_node_uuid': 'c2bd3223-7587-4bc5-9cae-794f61a80720',
│   'target_node_uuid': 'd1088975-8c10-436a-a40a-e613cb6554d3',
│   'created_at': datetime.datetime(2026, 7, 27, 17, 13, 31, 604852, tzinfo=<UTC>),
│   'name': 'SOLD_BY',
│   'fact': 'TinyBirds Wool Runners - Little Kids - Natural Black (Blizzard Sole) are sold by Manybirds',
│   'episodes': ['938ad52d-0b57-4496-8a2d-e54ade1abe2a'],
│   'expired_at': None,
│   'valid_at': datetime.datetime(2026, 7, 27, 17, 13, 20, 358357, tzinfo=<UTC>),
│   'invalid_at': None,
│   'reference_time': None,
│   'attributes': {}
}

### Ingest Conversation 1
Feeds the first scripted shoe conversation (`shoe_conversation_1`) into the graph via `add_messages`, introducing "John" and his shoe purchase.

In [12]:
await add_messages(client, shoe_conversation_1, prefix='conversation-1')

### Search: John's Shoe Size
Searches the graph for John's shoe size to confirm the conversation episodes were extracted and linked correctly.

In [13]:
r = await client.search("What is John's shoe size?", num_results=2)

pretty_print(r)

[
│   {
│   │   'uuid': '023ceae4-6e10-47c1-9e92-24a252569ac3',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '9e698486-f6c2-4373-b977-b54e5510ceb2',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 56, 686744, tzinfo=<UTC>),
│   │   'name': 'IS_ALLERGIC_TO',
│   │   'fact': 'John is allergic to wool',
│   │   'episodes': ['8435882f-3d29-4a5a-9ea3-40bcb6b130c7'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 3, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': 'ba32eb90-cab8-4d71-8dbf-e6d9be95c7d9',
│   │   'group_id': '',
│   │   'source_node_uuid': 'a20839e6-faa9-49ae-8261-d9ebf97f8aaf',
│   │   'target_node_uuid': '33d08a70-4568-46b5-81ea-82a87431ae9d',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 15, 852369, tzinfo=<UTC>),
│   │   'name': 'IS',
│   │   'fact': 'SalesBot is ManyBirds Assistant',
│   │   'episodes': ['b472a1af-9db9-47f4-a897-b4ca1a83ae7d'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 0, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   }
]

### Look Up John's Node UUID
Uses the lower-level `client._search` with the `NODE_HYBRID_SEARCH_RRF` recipe to find the entity node for "John" and capture its UUID (`john_uuid`) — needed by the node-distance-reranked searches later. Raises immediately if no "John" node was extracted, instead of failing confusingly further down.

In [10]:
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

nl = await client._search('John', NODE_HYBRID_SEARCH_RRF)

nl = nl.nodes

if not nl:
    raise ValueError("No nodes returned for query 'John'")

pretty_print(nl[0])

john_uuid = nl[0].uuid

EntityNode(
│   uuid='4b609d09-6358-4234-9a32-f582ff4befe8',
│   name='John',
│   group_id='',
│   labels=['Entity'],
│   created_at=datetime.datetime(2026, 7, 27, 17, 16, 29, 232979, tzinfo=<UTC>),
│   name_embedding=None,
│   summary="John is looking for a new pair of shoes as of 2024-07-30.\nJohn is allergic to wool\nSalesBot assists John\nJohn needs to return the Men's Couriers he bought recently\nJohn finds the Men's Couriers too tight for his wide feet\nJohn bought the Men's Couriers",
│   attributes={}
)

### Search: Standard RRF Reranking
Searches whether John can wear ManyBirds Wool Runners using plain Reciprocal Rank Fusion (no node-distance bias) and prints each returned fact.

In [8]:
r = await client.search('Can John wear ManyBirds Wool Runners?', num_results=3)

print('-' * 100)
print('Standard Reciprocal Rank Fusion Reranking')
print('-' * 100)
for record in r:
    print(record.fact)

----------------------------------------------------------------------------------------------------
Standard Reciprocal Rank Fusion Reranking
----------------------------------------------------------------------------------------------------
Men's SuperLight Wool Runners - Dark Grey (Medium Grey Sole) is sold by Manybirds
TinyBirds Wool Runners - Little Kids - Natural Black (Blizzard Sole) are sold by Manybirds
John is allergic to wool


### Search: Node-Distance Reranking from John
Same query as above, but reranked by graph proximity to John's node (`center_node_uuid=john_uuid`) — compare the ordering against the previous cell to see the effect of node-distance reranking.

In [11]:
r = await client.search(
    'Can John wear ManyBirds Wool Runners?', center_node_uuid=john_uuid, num_results=3
)

print('-' * 100)
print("Node Distance Reranking from 'John' node")
print('-' * 100)
for record in r:
    print(record.fact)

----------------------------------------------------------------------------------------------------
Node Distance Reranking from 'John' node
----------------------------------------------------------------------------------------------------
John is allergic to wool
John finds the Men's Couriers too tight for his wide feet
SalesBot assists John


### Ingest Conversation 2
Feeds the second scripted conversation (`shoe_conversation_2`, where John returns his shoes) into the graph.

In [17]:
await add_messages(client, shoe_conversation_2, prefix='conversation-2')

### Search: What Shoes Has John Purchased? (top 3)
Node-distance-reranked search for John's shoe purchases, limited to 3 results.

In [12]:
r = await client.search('What shoes has John purchased?', center_node_uuid=john_uuid, num_results=3)

pretty_print(r)

[
│   {
│   │   'uuid': '71d279df-dfa3-4e96-82bd-26b0eac36470',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696150, tzinfo=<UTC>),
│   │   'name': 'FINDS_TOO_TIGHT_FOR',
│   │   'fact': "John finds the Men's Couriers too tight for his wide feet",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5d515ee8-eed2-4d44-bf0d-082400b771db',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696016, tzinfo=<UTC>),
│   │   'name': 'NEEDS_TO_RETURN',
│   │   'fact': "John needs to return the Men's Couriers he bought recently",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '023ceae4-6e10-47c1-9e92-24a252569ac3',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '9e698486-f6c2-4373-b977-b54e5510ceb2',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 56, 686744, tzinfo=<UTC>),
│   │   'name': 'IS_ALLERGIC_TO',
│   │   'fact': 'John is allergic to wool',
│   │   'episodes': ['8435882f-3d29-4a5a-9ea3-40bcb6b130c7'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 3, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   }
]

### Search: What Shoes Has John Purchased? (top 5)
Same query as above with `num_results=5` — compare against the top-3 version to see how the extra results add context (e.g., the return).

In [13]:
r = await client.search('What shoes has John purchased?', center_node_uuid=john_uuid, num_results=5)

pretty_print(r)

[
│   {
│   │   'uuid': '71d279df-dfa3-4e96-82bd-26b0eac36470',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696150, tzinfo=<UTC>),
│   │   'name': 'FINDS_TOO_TIGHT_FOR',
│   │   'fact': "John finds the Men's Couriers too tight for his wide feet",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5d515ee8-eed2-4d44-bf0d-082400b771db',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696016, tzinfo=<UTC>),
│   │   'name': 'NEEDS_TO_RETURN',
│   │   'fact': "John needs to return the Men's Couriers he bought recently",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '023ceae4-6e10-47c1-9e92-24a252569ac3',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '9e698486-f6c2-4373-b977-b54e5510ceb2',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 56, 686744, tzinfo=<UTC>),
│   │   'name': 'IS_ALLERGIC_TO',
│   │   'fact': 'John is allergic to wool',
│   │   'episodes': ['8435882f-3d29-4a5a-9ea3-40bcb6b130c7'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 3, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5b0d0198-9816-4b6f-8b22-8c7b9cad82df',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696185, tzinfo=<UTC>),
│   │   'name': 'BOUGHT',
│   │   'fact': "John bought the Men's Couriers",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 5, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '1ed4fc23-013b-43d0-ad61-24901e860c0b',
│   │   'group_id': '',
│   │   'source_node_uuid': 'a20839e6-faa9-49ae-8261-d9ebf97f8aaf',
│   │   'target_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 14, 372365, tzinfo=<UTC>),
│   │   'name': 'ASSISTS',
│   │   'fact': 'SalesBot assists John',
│   │   'episodes': ['70aa9955-3e9b-40b0-b446-0cc424d39a85'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 0, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   }
]

### Search: Who Is John?
A broad, open-ended search to see what the graph has aggregated about John across both conversations.

In [14]:
r = await client.search('Who is John?', num_results=5)

pretty_print(r)

[
│   {
│   │   'uuid': '1ed4fc23-013b-43d0-ad61-24901e860c0b',
│   │   'group_id': '',
│   │   'source_node_uuid': 'a20839e6-faa9-49ae-8261-d9ebf97f8aaf',
│   │   'target_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 14, 372365, tzinfo=<UTC>),
│   │   'name': 'ASSISTS',
│   │   'fact': 'SalesBot assists John',
│   │   'episodes': ['70aa9955-3e9b-40b0-b446-0cc424d39a85'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 0, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': 'ba32eb90-cab8-4d71-8dbf-e6d9be95c7d9',
│   │   'group_id': '',
│   │   'source_node_uuid': 'a20839e6-faa9-49ae-8261-d9ebf97f8aaf',
│   │   'target_node_uuid': '33d08a70-4568-46b5-81ea-82a87431ae9d',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 15, 852369, tzinfo=<UTC>),
│   │   'name': 'IS',
│   │   'fact': 'SalesBot is ManyBirds Assistant',
│   │   'episodes': ['b472a1af-9db9-47f4-a897-b4ca1a83ae7d'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 0, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '023ceae4-6e10-47c1-9e92-24a252569ac3',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '9e698486-f6c2-4373-b977-b54e5510ceb2',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 56, 686744, tzinfo=<UTC>),
│   │   'name': 'IS_ALLERGIC_TO',
│   │   'fact': 'John is allergic to wool',
│   │   'episodes': ['8435882f-3d29-4a5a-9ea3-40bcb6b130c7'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 3, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5d515ee8-eed2-4d44-bf0d-082400b771db',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696016, tzinfo=<UTC>),
│   │   'name': 'NEEDS_TO_RETURN',
│   │   'fact': "John needs to return the Men's Couriers he bought recently",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5b0d0198-9816-4b6f-8b22-8c7b9cad82df',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696185, tzinfo=<UTC>),
│   │   'name': 'BOUGHT',
│   │   'fact': "John bought the Men's Couriers",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 5, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   }
]

### Search: John's Discomfort with the Couriers
Searches specifically for what John did about his fit complaint — should surface the return episode from conversation 2.

In [16]:
r = await client.search(
    'What did John do about his discomfort with the Mens Couriers shoes', num_results=5
)

pretty_print(r)

[
│   {
│   │   'uuid': '71d279df-dfa3-4e96-82bd-26b0eac36470',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696150, tzinfo=<UTC>),
│   │   'name': 'FINDS_TOO_TIGHT_FOR',
│   │   'fact': "John finds the Men's Couriers too tight for his wide feet",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5b0d0198-9816-4b6f-8b22-8c7b9cad82df',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696185, tzinfo=<UTC>),
│   │   'name': 'BOUGHT',
│   │   'fact': "John bought the Men's Couriers",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 5, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '5d515ee8-eed2-4d44-bf0d-082400b771db',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '6efaf02b-3d38-4502-af2c-b4435f99f70e',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 24, 696016, tzinfo=<UTC>),
│   │   'name': 'NEEDS_TO_RETURN',
│   │   'fact': "John needs to return the Men's Couriers he bought recently",
│   │   'episodes': ['d0aa7005-d5e9-49f3-baa0-46cfe2707ae0'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 1, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '023ceae4-6e10-47c1-9e92-24a252569ac3',
│   │   'group_id': '',
│   │   'source_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'target_node_uuid': '9e698486-f6c2-4373-b977-b54e5510ceb2',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 16, 56, 686744, tzinfo=<UTC>),
│   │   'name': 'IS_ALLERGIC_TO',
│   │   'fact': 'John is allergic to wool',
│   │   'episodes': ['8435882f-3d29-4a5a-9ea3-40bcb6b130c7'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 7, 30, 0, 3, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   },
│   {
│   │   'uuid': '1ed4fc23-013b-43d0-ad61-24901e860c0b',
│   │   'group_id': '',
│   │   'source_node_uuid': 'a20839e6-faa9-49ae-8261-d9ebf97f8aaf',
│   │   'target_node_uuid': '4b609d09-6358-4234-9a32-f582ff4befe8',
│   │   'created_at': datetime.datetime(2026, 7, 27, 17, 20, 14, 372365, tzinfo=<UTC>),
│   │   'name': 'ASSISTS',
│   │   'fact': 'SalesBot assists John',
│   │   'episodes': ['70aa9955-3e9b-40b0-b446-0cc424d39a85'],
│   │   'expired_at': None,
│   │   'valid_at': datetime.datetime(2024, 8, 20, 0, 0, tzinfo=<UTC>),
│   │   'invalid_at': None,
│   │   'reference_time': None,
│   │   'attributes': {}
│   }
]